In [6]:
import random
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from mlsmote import get_tail_label, get_minority_instace, MLSMOTE

In [7]:
df = pd.read_csv("../cleaned_animes.csv")

df["text"] = df["title_english"] + " " + df["synopsis"]
# df["genre_list"] = df["genres"].apply(lambda x: x.split("|"))

# turn each list of genres into a row of 0s and 1s
import ast
from sklearn.preprocessing import MultiLabelBinarizer

df = pd.read_csv("../cleaned_animes.csv")

df["text"] = df["title_english"] + " " + df["synopsis"]

# Convert strings into actual lists
df["genre_list"] = df["genre_list"].apply(ast.literal_eval)

# Binarize genres
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(df["genre_list"])

print("Number of shows:", len(df))
print("Number of genres:", len(mlb.classes_))
print(mlb.classes_)

Number of shows: 10523
Number of genres: 20
['Action' 'Adventure' 'Avant Garde' 'Boys Love' 'Comedy' 'Drama' 'Ecchi'
 'Erotica' 'Fantasy' 'Girls Love' 'Gourmet' 'Hentai' 'Horror' 'Mystery'
 'Romance' 'Sci-Fi' 'Slice of Life' 'Sports' 'Supernatural' 'Suspense']


In [8]:
print(df["genre_list"].iloc[0])
print(type(df["genre_list"].iloc[0]))
print(mlb.classes_)

['Action', 'Sci-Fi']
<class 'list'>
['Action' 'Adventure' 'Avant Garde' 'Boys Love' 'Comedy' 'Drama' 'Ecchi'
 'Erotica' 'Fantasy' 'Girls Love' 'Gourmet' 'Hentai' 'Horror' 'Mystery'
 'Romance' 'Sci-Fi' 'Slice of Life' 'Sports' 'Supernatural' 'Suspense']


In [9]:
X_train, X_test, y_train, y_test = train_test_split(df["text"], Y, test_size=0.2, random_state=1001)

# TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, stop_words="english", ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# shrink 5000 word columns down to 100 features
svd = TruncatedSVD(n_components=100, random_state=1001)
X_train_svd = svd.fit_transform(X_train_tfidf)
X_test_svd = svd.transform(X_test_tfidf)

print("Training shape:", X_train_svd.shape)
print("Testing shape:", X_test_svd.shape)

Training shape: (8418, 100)
Testing shape: (2105, 100)


In [10]:
# mlsmote.py works with pandas DataFrames
X_train_df = pd.DataFrame(X_train_svd)
y_train_df = pd.DataFrame(y_train, columns=mlb.classes_)

In [11]:
tail_genres = get_tail_label(y_train_df)
print("Rare genres:", tail_genres)

X_sub, y_sub = get_minority_instace(X_train_df, y_train_df)
print("Training shows with at least one rare genre:", len(X_sub))

Rare genres: ['Boys Love', 'Erotica', 'Girls Love', 'Gourmet']
Training shows with at least one rare genre: 261


In [12]:
#double each rare genre 
n_fake = int(y_train_df[tail_genres].sum().sum())
print("Fake shows to make:", n_fake)

Fake shows to make: 282


In [13]:
random.seed(1001)  # mlsmote.py uses Python's random module, so set its seed
X_res, y_res = MLSMOTE(X_sub, y_sub, n_fake)

# keep only the new fake shows (the last n_fake rows)
X_fake_ml = X_res.iloc[-n_fake:].to_numpy()
y_fake_ml = y_res.iloc[-n_fake:].to_numpy()

# add them to the full training set
X_ml = np.vstack([X_train_svd, X_fake_ml])
Y_ml = np.vstack([y_train, y_fake_ml])
print("Training shows after MLSMOTE:", len(X_ml))

Training shows after MLSMOTE: 8700


In [14]:
def gaussian_mlsmote(X, Y, tail_genres, alpha=0.5, random_state=1001):
    rng = np.random.default_rng(random_state)
    new_X = []
    new_Y = []

    for genre in tail_genres:
        j = list(mlb.classes_).index(genre)
        X_genre = X[Y[:, j] == 1]
        Y_genre = Y[Y[:, j] == 1]
        n_new = len(X_genre) # one fake show per real show (doubles the genre)

        # global stats for this genre
        genre_mean = X_genre.mean(axis=0)
        genre_std = X_genre.std(axis=0)

        # draw the fake shows from a normal distribution
        X_fake = rng.normal(
            loc=genre_mean,
            scale=alpha * np.maximum(genre_std, 1e-6),
            size=(n_new, X.shape[1])
        )

        # find the 5 closest real shows (with this genre) to each fake show
        nn = NearestNeighbors(n_neighbors=5).fit(X_genre)
        _, neighbor_index = nn.kneighbors(X_fake)

        for f in range(n_new):
            # count how many of the 5 neighbors have each genre... keep genres with 3 or more
            votes = Y_genre[neighbor_index[f]].sum(axis=0)
            y_fake = (votes > 2).astype(int)

            new_X.append(X_fake[f])
            new_Y.append(y_fake)

    return np.array(new_X), np.array(new_Y)

In [15]:
X_fake_gm, y_fake_gm = gaussian_mlsmote(X_train_svd, y_train, tail_genres)

X_gm = np.vstack([X_train_svd, X_fake_gm])
Y_gm = np.vstack([y_train, y_fake_gm])
print("Training shows after Gaussian MLSMOTE:", len(X_gm))

Training shows after Gaussian MLSMOTE: 8700


In [16]:
def train_and_predict(X_train, Y_train, X_test, threshold=0.3):
    predictions = np.zeros((X_test.shape[0], Y_train.shape[1]))

    for j in range(Y_train.shape[1]):
        model = LogisticRegression(max_iter=2000)
        model.fit(X_train, Y_train[:, j])
        predictions[:, j] = (model.predict_proba(X_test)[:, 1] >= threshold).astype(int)

    return predictions

In [17]:
pred_baseline = train_and_predict(X_train_svd, y_train, X_test_svd) #no oversampling
pred_mlsmote = train_and_predict(X_ml, Y_ml, X_test_svd) # MLSMOTE
pred_gaussian_mlsmote = train_and_predict(X_gm, Y_gm, X_test_svd)# Gaussian MLSMOTE

In [19]:
results = pd.DataFrame({
    "Method": ["Baseline (no oversampling)", "MLSMOTE", "Gaussian MLSMOTE"],
    "Macro F1": [
        f1_score(y_test, pred_baseline, average="macro", zero_division=0),
        f1_score(y_test, pred_mlsmote, average="macro", zero_division=0),
        f1_score(y_test, pred_gaussian_mlsmote, average="macro", zero_division=0),
    ],
    "Micro F1": [
        f1_score(y_test, pred_baseline, average="micro", zero_division=0),
        f1_score(y_test, pred_mlsmote, average="micro", zero_division=0),
        f1_score(y_test, pred_gaussian_mlsmote, average="micro", zero_division=0),
    ],
})
results.round(4)

,Method,Macro F1,Micro F1
0,Baseline (no oversampling),0.2601,0.5210
1,MLSMOTE,0.2552,0.5182
2,Gaussian MLSMOTE,0.2574,0.5193
